# Scaling Adherence Analysis

Verifies that **asymmetric scaling** is working: strong signals front-load, weak signals spread evenly.

- Strong (≥0.70): `front_load_factor=2.0`
- Moderate (≥0.30): `front_load_factor=1.3`
- Weak (<0.30): `front_load_factor=1.0` (uniform)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

DATA_DIR           = './output'
SCALING_DAYS       = 5
STRONG_THRESHOLD   = 0.70
MODERATE_THRESHOLD = 0.30

def build_scaling_schedule(n_days, front_load_factor=1.0):
    raw = [front_load_factor ** i for i in range(n_days)]
    total = sum(raw); cumulative = 0.0; schedule = []
    for v in raw:
        cumulative += v / total
        schedule.append(round(min(cumulative, 1.0), 6))
    return schedule

strong_sched   = build_scaling_schedule(SCALING_DAYS, 2.0)
moderate_sched = build_scaling_schedule(SCALING_DAYS, 1.3)
weak_sched     = build_scaling_schedule(SCALING_DAYS, 1.0)
sched_map      = {'strong': strong_sched, 'moderate': moderate_sched, 'weak': weak_sched}

print('Strong   schedule:', [f'{v:.3f}' for v in strong_sched])
print('Moderate schedule:', [f'{v:.3f}' for v in moderate_sched])
print('Weak     schedule:', [f'{v:.3f}' for v in weak_sched])

In [ ]:
tgt = pd.read_csv(f'{DATA_DIR}/targets.csv', parse_dates=['date'])
print(f'Targets: {len(tgt):,} rows | columns: {list(tgt.columns)}')
tgt.head(3)

In [ ]:
# Bucket by tier from signal_strength column
def tier(s):
    if pd.isna(s): return 'unknown'
    if s >= STRONG_THRESHOLD:   return 'strong'
    if s >= MODERATE_THRESHOLD: return 'moderate'
    return 'weak'

tgt['tier'] = tgt['signal_strength'].apply(tier)
print('Tier distribution:')
print(tgt['tier'].value_counts())

In [ ]:
# Compute expected fraction for each row and compare to scheduled_fraction
def expected_fraction(row):
    sched = sched_map.get(row['tier'])
    if sched is None: return np.nan
    day = min(int(row.get('scale_day', 0)), len(sched)-1)
    return sched[day]

scaling = tgt[tgt['is_scaling'] == True].copy()
scaling['expected_fraction'] = scaling.apply(expected_fraction, axis=1)
scaling['fraction_error']    = scaling['scheduled_fraction'] - scaling['expected_fraction']

print(f'Scaling rows      : {len(scaling):,}')
print(f'Mean |error|      : {scaling["fraction_error"].abs().mean():.6f}')
print(f'Max  |error|      : {scaling["fraction_error"].abs().max():.6f}')
print(f'Rows with error>0 : {(scaling["fraction_error"].abs() > 1e-5).sum()}')

In [ ]:
# Average scheduled fraction by tier and scale_day vs expected
grp = scaling.groupby(['tier','scale_day']).agg(
    scheduled_fraction=('scheduled_fraction','mean'),
    n=('scheduled_fraction','count')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors = {'strong':'steelblue', 'moderate':'darkorange', 'weak':'green'}

for ax, t in zip(axes, ['strong','moderate','weak']):
    sub = grp[grp['tier']==t].sort_values('scale_day')
    expected = sched_map[t]
    ax.plot(range(len(expected)), expected, 'k--', lw=2, label='Expected')
    if len(sub):
        ax.plot(sub['scale_day'], sub['scheduled_fraction'], 'o-',
                color=colors[t], lw=2, ms=8, label='Observed')
    ax.set_title(f'{t.capitalize()} Tier (≥{STRONG_THRESHOLD if t=="strong" else MODERATE_THRESHOLD if t=="moderate" else 0})',
                 fontsize=12)
    ax.set_xlabel('Scale Day (0=rebalance day)')
    ax.set_ylabel('Fraction of Final Weight')
    ax.set_ylim(0, 1.1); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Scaling Adherence: Observed vs Expected Fraction by Signal Tier',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/scaling_adherence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Classification breakdown
if 'classification' in tgt.columns:
    print('Classification counts:')
    print(tgt['classification'].value_counts())

    flips = tgt[tgt['classification']=='FLIP']
    print(f'\nFLIP rows with non-zero scheduled_w: {(flips["scheduled_w"].abs() > 1e-6).sum()}  (should be 0)')
    print(f'HOLD rows total: {(tgt["classification"]=="HOLD").sum()}')

In [ ]:
# Error distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(scaling['fraction_error'].dropna()*100, bins=60,
        color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', lw=1.5, ls='--', label='Zero error')
ax.set_xlabel('Scheduled − Expected Fraction (percentage points)')
ax.set_title('Scaling Fraction Error Distribution', fontsize=12)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Adherence rate (|error| < 0.001): {(scaling["fraction_error"].abs() < 0.001).mean()*100:.1f}%')